# NB_Purview_Copilot_SilverToGold
**Source:** `AI_Medallion.Silver.purview_copilot_usage` (OneLake **or** local merge staging from `NB_Purview_Copilot_BronzeToSilver`)  
**Target:** `AI_Medallion.Gold.fact_copilot_active`  
**Grain:** `GROUP BY` Okta user, activity date, interaction type — `Request` = `SUM(prompt_count)`  
**Filter:** groups with `SUM(prompt_count) >= 1` (all license types)  
**Load:** Incremental merge on `RecordSurrogateKey`  
**Run after:** `NB_Purview_Copilot_BronzeToSilver`

In [ ]:
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")

# local_staging = read ~/.fabric/merge_staging/… from BronzeToSilver (no OneLake silver read)
PURVIEW_SILVER_SOURCE = "local_staging"
if PURVIEW_SILVER_SOURCE == "local_staging":
    os.environ["MEDALLION_FRESH_SPARK"] = "1"
os.environ.setdefault("PYSPARK_DRIVER_MEMORY", "8g")
os.environ["MEDALLION_STAGING_SPARK_PARQUET"] = "0"
os.environ.setdefault("MEDALLION_STAGING_COLLECT_ROWS", "15000")


In [ ]:
%run ./Common/bootstrap


In [ ]:
# --- Purview gold local silver (notebook helpers) ---
import os
import shutil
from pathlib import Path


def _purview_merge_staging_dir(table_fqn: str) -> Path:
    return Path.home() / ".fabric" / "merge_staging" / table_fqn.replace(".", "__")


def _purview_spark_is_alive(session) -> bool:
    if session is None:
        return False
    try:
        session.range(1).count()
        return True
    except Exception:
        return False


def _purview_recreate_spark():
    global spark
    from Common.bootstrap import _stop_active_spark, create_onelake_spark

    print("purview gold: recreating Spark session…", flush=True)
    _stop_active_spark()
    spark = create_onelake_spark("Medallion-Purview-Gold")
    ng = globals()
    ng["spark"] = spark
    reload_io_helpers(ng)
    apply_local_merge_spark_conf(spark)
    spark.range(1).count()
    return spark


def _purview_ensure_spark_alive():
    global spark
    if _purview_spark_is_alive(spark):
        return spark
    return _purview_recreate_spark()


def _purview_reaggregate_active_copilot(df):
    """Combine partial aggregates from batched silver parts."""
    keys = [
        "purview_copilot_usage_date",
        "okta_user_id",
        "purview_copilot_usage_interaction_type",
    ]
    return (
        df.groupBy(*keys)
        .agg(
            F.sum("Request").cast("int").alias("Request"),
            F.first("Month_No", ignorenulls=True).cast("int").alias("Month_No"),
            F.first("okta_user_full_name", ignorenulls=True).alias("okta_user_full_name"),
            F.first("okta_user_employee_number", ignorenulls=True).alias(
                "okta_user_employee_number"
            ),
            F.first("okta_user_email", ignorenulls=True).alias("okta_user_email"),
            F.first("okta_user_full_email", ignorenulls=True).alias("okta_user_full_email"),
            F.first("okta_user_region", ignorenulls=True).alias("okta_user_region"),
            F.first("okta_user_country", ignorenulls=True).alias("okta_user_country"),
            F.first("hc_business_unit", ignorenulls=True).alias("hc_business_unit"),
            F.first("hc_job_title", ignorenulls=True).alias("hc_job_title"),
            F.first("hc_okta_user_country", ignorenulls=True).alias("hc_okta_user_country"),
            F.first("hc_office", ignorenulls=True).alias("hc_office"),
            F.first("hc_talent_segment", ignorenulls=True).alias("hc_talent_segment"),
            F.first("hc_division", ignorenulls=True).alias("hc_division"),
            F.first("hc_business_unit_code", ignorenulls=True).alias("hc_business_unit_code"),
        )
        .filter(F.col("Request") >= 1)
    )


def purview_read_local_silver_staging_batched(table_fqn: str):
    """Read silver merge staging parquet (BronzeToSilver output), aggregate per part."""
    staging = _purview_merge_staging_dir(table_fqn)
    part_files = sorted(staging.glob("part-*.parquet"))
    if not part_files:
        raise FileNotFoundError(
            f"No local silver staging under {staging}. "
            "Run NB_Purview_Copilot_BronzeToSilver (local_csv) first."
        )
    print(
        f"purview gold: local silver {staging} ({len(part_files)} part(s), batched aggregate)",
        flush=True,
    )
    combined = None
    total = len(part_files)
    for batch_no, part_path in enumerate(part_files, start=1):
        print(f"purview gold silver batch {batch_no}/{total}: {part_path.name}", flush=True)
        last_exc = None
        for attempt in range(2):
            try:
                _purview_ensure_spark_alive()
                apply_local_merge_spark_conf(spark)
                silver_batch = spark.read.parquet(str(part_path))
                partial = purview_active_copilot_aggregate(silver_batch)
                combined = (
                    partial if combined is None else combined.unionByName(partial)
                )
                last_exc = None
                break
            except BaseException as exc:
                last_exc = exc
                print(f"  attempt {attempt + 1} failed ({type(exc).__name__})", flush=True)
                if not _purview_spark_is_alive(spark):
                    _purview_recreate_spark()
        if last_exc is not None:
            raise RuntimeError(f"purview gold failed on {part_path.name}") from last_exc
    assert combined is not None
    return _purview_reaggregate_active_copilot(combined)


def purview_publish_gold_merge_staging_notebook(df, table_fqn: str) -> Path:
    """PyArrow chunked staging for gold merge (no JVM write.parquet)."""
    from Common import io

    path = _purview_merge_staging_dir(table_fqn)
    collect_rows = max(5_000, int(os.environ.get("MEDALLION_STAGING_COLLECT_ROWS", "15000")))
    print(
        f"purview gold staging: {table_fqn} -> {path} "
        f"(collect_rows={collect_rows}, PyArrow)",
        flush=True,
    )
    last_exc = None
    for attempt in range(2):
        try:
            _purview_ensure_spark_alive()
            apply_local_merge_spark_conf(spark)
            if path.exists():
                shutil.rmtree(path, ignore_errors=True)
            staged = io._sanitize_df_for_export(df)
            io._write_df_via_local_iterator_chunks(staged, path)
            if not any(path.glob("*.parquet")):
                raise RuntimeError(f"No parquet written under {path}")
            print(f"purview gold staging: done -> {path}", flush=True)
            return path
        except BaseException as exc:
            last_exc = exc
            print(f"purview gold staging attempt {attempt + 1} failed", flush=True)
            if not _purview_spark_is_alive(spark):
                _purview_recreate_spark()
    raise RuntimeError(f"purview gold staging failed for {table_fqn}") from last_exc



In [ ]:
_purview_ensure_spark_alive()
apply_local_merge_spark_conf(spark)

if PURVIEW_SILVER_SOURCE == "local_staging":
    grouped = purview_read_local_silver_staging_batched(TABLE_PURVIEW_COPILOT_SILVER)
else:
    silver = read_table_broken_lineage(TABLE_PURVIEW_COPILOT_SILVER, "purview_copilot_silver")
    grouped = break_lineage_local(
        purview_active_copilot_aggregate(silver), "purview_active_copilot_daily"
    )

grouped = grouped.withColumn(
    FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY, purview_active_copilot_merge_key()
)



In [ ]:
apply_local_merge_spark_conf(spark)

use_case_multiplier = F.lit(COPILOT_USE_CASE_MULTIPLIER).cast("decimal(18,8)")
automation_poc = F.lit(COPILOT_AUTOMATION_POC).cast("decimal(18,8)")
total_hours_saved_poc = F.lit(COPILOT_TOTAL_HOURS_SAVED_POC).cast("decimal(18,8)")

interaction = F.col("purview_copilot_usage_interaction_type")

fact = grouped.select(
    F.col("Month_No"),
    F.col("Request"),
    F.lit(COPILOT_ASSET_ID).alias("assetId"),
    F.lit(COPILOT_ASSET_NAME).alias("assetName"),
    F.lit(None).cast("string").alias("eventType"),
    F.lit(None).cast("string").alias("id_project"),
    F.lit(None).cast("string").alias("project"),
    F.lit(None).cast("string").alias("account"),
    F.col("okta_user_id").alias("userId"),
    F.col("okta_user_full_name").alias("username"),
    F.col("okta_user_employee_number").alias("employeeid"),
    F.col("okta_user_email").alias("userEmail"),
    F.col("okta_user_full_email").alias("userFullEmail"),
    F.col("okta_user_region").alias("Division"),
    F.col("okta_user_country").alias("userCountry"),
    purview_copilot_use_case_code(interaction).alias("useCaseCode"),
    interaction.alias("useCaseDisplayName"),
    (
        F.col("Request").cast("decimal(18,8)") * total_hours_saved_poc * use_case_multiplier
    ).alias("UseCaseHoursSaved"),
    (
        F.col("Request").cast("decimal(18,8)") * (total_hours_saved_poc / automation_poc)
    ).alias("UseCaseTotalHours"),
    F.lit(1).cast("int").alias("UserWithActivity"),
    F.col("purview_copilot_usage_date").alias("LastModification"),
    F.col("hc_business_unit").alias("User_UN"),
    F.col("hc_job_title").alias("JobRoles"),
    F.col("hc_okta_user_country").alias("Employee_Country"),
    F.col("hc_office").alias("Employee_Office"),
    F.col("hc_talent_segment").alias("Segment"),
    F.col("hc_division").alias("Foundation"),
    F.col("hc_business_unit_code").alias("Employee_Business_Unit_Code"),
    F.lit(None).cast("string").alias("asset_model"),
    F.lit(None).cast("string").alias("asset_useCase"),
    F.lit(None).cast("string").alias("asset_mySourceType"),
    F.lit(None).cast("string").alias("asset_sourceType"),
    F.lit(None).cast("string").alias("billingClientName"),
    F.lit(None).cast("string").alias("billingCountry"),
    F.lit(None).cast("string").alias("billingRegion"),
    F.lit(None).cast("string").alias("flowid"),
    use_case_multiplier.alias("useCaseMultiplier"),
    automation_poc.alias("automation_poc"),
    total_hours_saved_poc.alias("totalHoursSaved_poc"),
    F.col(FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY),
)

fact = fact.filter(
    F.col("Month_No").isNotNull()
    & F.col("Request").isNotNull()
    & F.col("assetId").isNotNull()
    & F.col("assetName").isNotNull()
    & F.col("useCaseCode").isNotNull()
    & F.col("useCaseDisplayName").isNotNull()
    & F.col("UseCaseHoursSaved").isNotNull()
    & F.col("UseCaseTotalHours").isNotNull()
    & F.col("UserWithActivity").isNotNull()
    & F.col("LastModification").isNotNull()
    & F.col("useCaseMultiplier").isNotNull()
    & F.col("automation_poc").isNotNull()
    & F.col("totalHoursSaved_poc").isNotNull()
    & F.col(FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY).isNotNull()
)

if PURVIEW_SILVER_SOURCE == "local_staging":
    purview_publish_gold_merge_staging_notebook(fact, TABLE_FACT_COPILOT_ACTIVE)
else:
    publish_merge_staging(fact, TABLE_FACT_COPILOT_ACTIVE)



In [ ]:
merge_df = None if PURVIEW_SILVER_SOURCE == "local_staging" else fact
merge_incremental(
    spark,
    merge_df,
    TABLE_FACT_COPILOT_ACTIVE,
    FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY,
)
show_sample(fact.limit(10))
